<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/FineTuning_LLM-Mistral-7B-Instruct-v0.3_for-text-to-SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

v0.1:    https://medium.com/thedeephub/fine-tuning-the-llm-mistral-7b-for-text-to-sql-with-sql-create-context-dataset-4e9234f7691c?postPublishedType=repub

As of April 2026, the latest and most advanced version of the 7B instruct family is **Mistral-7B-Instruct-v0.3**.

While the original `v0.1` was a breakthrough for the 7B-parameter class, Mistral AI has released two major updates since then that significantly improved its capabilities.

### Major Versions Compared

| Feature | v0.1 (Original) | v0.2 | v0.3 (Latest) |
| :--- | :--- | :--- | :--- |
| **Context Window** | 8K tokens | 32K tokens | **32K tokens** |
| **Tokenizer** | v1 | v2 | **v3 (extended)** |
| **Vocabulary Size** | 32,000 | 32,000 | **32,768** |
| **Function Calling** | No | No | **Yes (Native)** |
| **Attention Mechanism** | Sliding Window | No SWA | **No SWA** |

---

### Why move to v0.3?

1.  **Native Function Calling:** Unlike `v0.1`, which required complex prompting or wrappers to interact with external tools, `v0.3` supports function calling natively. This is a game-changer if you're building agents or RAG systems.
2.  **Extended Vocabulary:** The v3 tokenizer increases the vocabulary size to 32,768, which makes it more efficient at handling different languages and specialized technical terms.
3.  **Removal of Sliding Window Attention (SWA):** While SWA was an interesting optimization in `v0.1`, `v0.3` follows the `v0.2` trend of removing it to improve performance consistency and simplify integration with various inference engines (like `vLLM` or `llama.cpp`).

### How to get it
You can find the official weights on Hugging Face under the identifier: `mistralai/Mistral-7B-Instruct-v0.3`.

If you're using **Ollama**, you can pull the latest version by simply running:
`ollama run mistral` (which usually defaults to the latest 7B tag).

Are you planning to deploy this locally on one of your edge devices, or are you looking to fine-tune it for a specific task?

In [ ]:
# Install Pytorch & other libraries
!pip install torch tensorboard --quiet

# Install Hugging Face libraries
!pip install  --upgrade transformers datasets accelerate evaluate bitsandbytes --quiet

#FlashAttention only supports Ampere GPUs or newer. #NEED A100 IN GOOGLE COLAB
#!pip install -U transformers
#!pip install -U flash-attn --no-build-isolation --quiet

!pip install -q https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

! pip install peft --quiet
! pip install datasets trl ninja packaging --quiet

# Uncomment only if you're using A100 GPU
#!pip install flash-attn --no-build-isolation
!pip install diffusers safetensors  --quiet
!pip install colab-env --quiet


In [ ]:
!pip install --upgrade trl transformers accelerate peft bitsandbytes -q

In [21]:
import colab_env
import os

access_token_write = os.environ.get("HUGGINGFACE_ACCESS_TOKEN_WRITE")

In [ ]:
print(access_token_write)

In [23]:
!ls

gdrive	sample_data


In [25]:
from huggingface_hub import login

import os
os.environ.pop("HF_TOKEN", None)

login(
  token=access_token_write,
  add_to_git_credential=True
)

In [26]:
import torch
import os
import sys
import json
import IPython
from datetime import datetime
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

In [28]:
# set device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!apt-get update && apt-get install -y cuda-11-0

In [32]:
!python --version
!nvcc --version
!nvidia-smi

Python 3.12.13
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Tue Apr 14 18:27:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   3

In [33]:
import torch
torch.__version__

'2.10.0+cu128'

In [1]:
import torch
import os
import sys
import json
import IPython
from datetime import datetime
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

In [ ]:
from datasets import load_dataset

# Convert dataset to OAI messages
system_message = """You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
{schema}"""

def create_conversation(sample):
  return {
    "messages": [
      {"role": "system", "content": system_message.format(schema=sample["context"])},
      {"role": "user", "content": sample["question"]},
      {"role": "assistant", "content": sample["answer"]}
    ]
  }

# Load dataset from the hub
dataset = load_dataset("b-mc2/sql-create-context", split="train")
dataset = dataset.shuffle().select(range(12500))

# Convert dataset to OAI messages
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=False)

# split dataset into 10,000 training samples and 2,500 test samples
dataset = dataset.train_test_split(test_size=2500/12500)

print(dataset["train"][345]["messages"])

# save datasets to disk
dataset["train"].to_json("train_dataset.json", orient="records")
dataset["test"].to_json("test_dataset.json", orient="records")

In [ ]:
from datasets import load_dataset

# Load jsonl data from disk for sql
dataset = load_dataset("json", data_files="train_dataset.json", split="train")
eval_dataset = load_dataset("json", data_files="test_dataset.json", split="train")

In [4]:
print(dataset[345]["messages"])

[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_72 (venue VARCHAR, score VARCHAR)', 'role': 'system'}, {'content': 'Where was the game played when the score was 1-4?', 'role': 'user'}, {'content': 'SELECT venue FROM table_name_72 WHERE score = "1-4"', 'role': 'assistant'}]


In [5]:
print(eval_dataset[345]["messages"])

[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_14 (location VARCHAR, home_ground VARCHAR)', 'role': 'system'}, {'content': 'Which Location has a Home Ground of Mong kok stadium?', 'role': 'user'}, {'content': 'SELECT location FROM table_name_14 WHERE home_ground = "mong kok stadium"', 'role': 'assistant'}]


https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import logging

# Set logging to error only for transformers and peft
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)



# 1. Device Setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Model ID (Mistral v0.3 is the current stable version)
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 3. BitsAndBytes Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    trust_remote_code=True
)

# 5. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# 6. Manual ChatML Setup (Replacing setup_chat_format)
# Define the ChatML tokens
chatml_tokens = {"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]}
tokenizer.add_special_tokens(chatml_tokens)

# Set the pad token (using unk_token or im_end is common for Mistral)
tokenizer.pad_token = "<|im_end|>"
tokenizer.padding_side = 'right'

# 7. Resize Embeddings
# Crucial: This accommodates the new <|im_start|> and <|im_end|> tokens
model.resize_token_embeddings(len(tokenizer))

# 8. Define the Jinja Template for ChatML
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

print(f"Successfully loaded {model_id} on {device}")
print(f"Tokenizer size updated to: {len(tokenizer)}")

In [7]:
print(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32770, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [8]:
from peft import LoraConfig

# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
        lora_alpha=128,
        lora_dropout=0.05,
        r=256,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
)


Administration routines for huggingface.co

In [ ]:
#!pip install huggingface_hub --quiet
from huggingface_hub import HfApi

api = HfApi()
api.get_token_permission(token=access_token_write)
#api.set_access_token(access_token)
# frankmorales2020/Mistral-7B-text-to-sql Good


repo_id = "my-awesome-model-poc"

#frankmorales2020/Mistral-7B-squad2
#api.create_repo(repo_id=repo_id, private=False)

#api.delete_repo(repo_id=repo_id)


#api.upload_folder(
#    folder_path="./model",
#    repo_id=repo_id,
#    repo_type="model",
#)


In [ ]:
from trl import SFTTrainer, SFTConfig

import logging

# Set logging to error only for transformers and peft
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)


# 1. Define SFTConfig
sft_config = SFTConfig(
    output_dir="Mistral-7B-v0dot3-text-to-sql-flash-attention-2",
    push_to_hub=True,
    report_to="tensorboard",

    # --- Evaluation Logic (The Fix) ---
    eval_strategy="steps",          # REPLACES evaluation_strategy
    eval_steps=25,                  # How often to compute eval loss
    do_eval=True,

    # --- SFT Specifics ---
    max_length=3072,
    packing=True,
    dataset_text_field="text",

    # --- Training Hyperparameters ---
    num_train_epochs=3,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,   # Added for evaluation
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    warmup_steps=100,
    max_grad_norm=0.3,

    # --- Precision ---
    bf16=True,
    tf32=True,
    logging_steps=25,
    save_strategy="epoch"
)

# 2. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

# 3. Final Token Alignment (Based on your update)
# This ensures the model's generation config matches your new pad_token_id
model.config.pad_token_id = 32769
model.generation_config.pad_token_id = 32769

In [ ]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()

In [ ]:
# After trainer.train() finishes
trainer.save_model("Mistral-7B-v0dot3-text-to-sql-final")
tokenizer.save_pretrained("Mistral-7B-v0dot3-text-to-sql-final")

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

## Test Model and run Inference

In [ ]:
import subprocess
import sys

# Install a newer PEFT version that has the config parameters but hasn't introduced torchao dependency yet
print("Installing PEFT 0.9.0...")
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "peft", "-y"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "peft==0.9.0"])

# Also upgrade transformers to match
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.38.2", "--upgrade"])

print("\n✓ PEFT and transformers upgraded!")
print("⚠️  Please RESTART the kernel and then run the loading code below.")

In [ ]:
import os
os.environ["PEFT_DISABLE_TORCHAO"] = "1"

# Suppress all warnings
import warnings
warnings.filterwarnings("ignore")
from transformers import logging
logging.set_verbosity_error()
import huggingface_hub
huggingface_hub.logging.set_verbosity_error()

import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
peft_model_id = "./Mistral-7B-v0dot3-text-to-sql-flash-attention-2"

# Load tokenizer from base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Add the 2 extra tokens that were added during training
special_tokens = {"additional_special_tokens": ["<EXTRA_0>", "<EXTRA_1>"]}
tokenizer.add_special_tokens(special_tokens)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load and clean the adapter config
config_path = f"{peft_model_id}/adapter_config.json"
with open(config_path, 'r') as f:
    config_dict = json.load(f)

valid_params = ['r', 'lora_alpha', 'target_modules', 'lora_dropout', 'bias', 'task_type']
cleaned_config = {k: v for k, v in config_dict.items() if k in valid_params}
peft_config = LoraConfig(**cleaned_config)

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2"
)

# Resize embeddings
print(f"Resizing embeddings to {len(tokenizer)}...")
base_model.resize_token_embeddings(len(tokenizer))

# Load adapter
print("Loading PEFT adapter...")
model = PeftModel.from_pretrained(base_model, peft_model_id, config=peft_config)
model.eval()

print("✓ Model loaded successfully!\n")

def clean_sql_query(sql_text):
    """Clean up generated SQL query"""
    # Remove any "Assistant:" or "User:" prefixes
    sql_text = re.sub(r'^(Assistant|User):\s*', '', sql_text)

    # Stop at common patterns that indicate repetition
    stop_patterns = [
        r'\nächst', r'unächst', r'assistant', r'User:', r'\/\*+\*\/',
        r'SELECT.*SELECT'
    ]

    for pattern in stop_patterns:
        match = re.search(pattern, sql_text, re.IGNORECASE)
        if match:
            sql_text = sql_text[:match.start()]
            break

    # Remove excessive LIMIT values
    limit_pattern = r'LIMIT\s+(\d{10,})'
    if re.search(limit_pattern, sql_text):
        sql_text = re.sub(limit_pattern, 'LIMIT 100', sql_text)

    # Remove garbage words
    sql_text = re.sub(r'\s+(först|unächst|gennaio|Initially)\s+', ' ', sql_text)

    # Ensure SQL ends with semicolon
    sql_text = sql_text.strip()
    if sql_text and not sql_text.endswith(';'):
        sql_text += ';'

    return sql_text

def text_to_sql(question, schema, max_new_tokens=100):
    """Convert natural language question to SQL query"""
    system_message = f"""You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
{schema}"""

    prompt = f"""{system_message}

User: {question}
Assistant: """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    response = clean_sql_query(response)
    return response

print("--- Examples ---\n")

# Example 1
schema1 = """CREATE TABLE users (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100),
    created_at DATE
);"""
question1 = "Get all users"
result1 = text_to_sql(question1, schema1, max_new_tokens=50)
print(f"Q: {question1}")
print(f"SQL: {result1}\n")

# Example 2
schema2 = """CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(200),
    price DECIMAL(10,2),
    category VARCHAR(50)
);"""
question2 = "Products under 100 dollars"
result2 = text_to_sql(question2, schema2, max_new_tokens=50)
print(f"Q: {question2}")
print(f"SQL: {result2}\n")

# Example 3
schema3 = """CREATE TABLE employees (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(100),
    department VARCHAR(50),
    salary INT
);"""
question3 = "Highest paid employee"
result3 = text_to_sql(question3, schema3, max_new_tokens=60)
print(f"Q: {question3}")
print(f"SQL: {result3}\n")

print("✓ All done!")

Let’s load our test dataset try to generate an instruction.

In [9]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32770, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralFlashAttention2(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=256, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=256, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(

In [ ]:
import os
os.environ["PEFT_DISABLE_TORCHAO"] = "1"

# Suppress all warnings
import warnings
warnings.filterwarnings("ignore")
from transformers import logging
logging.set_verbosity_error()
import huggingface_hub
huggingface_hub.logging.set_verbosity_error()

import json
import torch
import re
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig
import random
import numpy as np

# --- Reproducibility ---

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
peft_model_id = "./Mistral-7B-v0dot3-text-to-sql-flash-attention-2"

def set_reproducibility(seed=123):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔐 H2E Determinism Locked | Seed: {seed}")


set_reproducibility(123)

# Load tokenizer from base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Add the 2 extra tokens
special_tokens = {"additional_special_tokens": ["<EXTRA_0>", "<EXTRA_1>"]}
tokenizer.add_special_tokens(special_tokens)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load and clean the adapter config
config_path = f"{peft_model_id}/adapter_config.json"
with open(config_path, 'r') as f:
    config_dict = json.load(f)

valid_params = ['r', 'lora_alpha', 'target_modules', 'lora_dropout', 'bias', 'task_type']
cleaned_config = {k: v for k, v in config_dict.items() if k in valid_params}
peft_config = LoraConfig(**cleaned_config)

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2"
)

# Resize embeddings
print(f"Resizing embeddings to {len(tokenizer)}...")
base_model.resize_token_embeddings(len(tokenizer))

# Load adapter
print("Loading PEFT adapter...")
model = PeftModel.from_pretrained(base_model, peft_model_id, config=peft_config)
model.eval()

print("✓ Model loaded successfully!\n")

In [2]:
import re
import torch
import sqlglot
from sqlglot import exp
from tqdm import tqdm
from datasets import load_dataset

class TextToSQLPipeline:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True):
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt
        )

    def __call__(self, prompt, max_new_tokens=256, do_sample=False, temperature=0.1,
                 top_k=50, top_p=0.95, eos_token_id=None, pad_token_id=None):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=do_sample,
                top_k=top_k,
                top_p=top_p,
                pad_token_id=pad_token_id or self.tokenizer.pad_token_id,
                eos_token_id=eos_token_id or self.tokenizer.eos_token_id
            )

        generated_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return [{"generated_text": prompt + generated_text}]

def extract_sql(text):
    """Extracts SQL and handles hallucinated noise."""
    patterns = [
        r'(SELECT\s+.*?)(?:;|$|unächst|först|user|assistant)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            sql = match.group(1).strip()
            sql = " ".join(sql.split())
            return sql
    return ""

def logical_norm(sql_string):
    if not sql_string or len(sql_string) < 5:
        return ""
    try:
        # 1. Parse SQL into a logical tree
        tree = sqlglot.parse_one(sql_string.lower())

        # 2. Fix Case 1 & 2: Iterate through all Literals (values)
        for node in tree.find_all(exp.Literal):
            # .this is the actual content (e.g., " 69" or 69)
            # strip() handles the " charles" leading space
            # str() + Literal.string() handles the 69 vs "69" type mismatch
            clean_val = str(node.this).strip()
            node.replace(exp.Literal.string(clean_val))

        # 3. Handle Table/Column names just in case they have spaces
        for node in tree.find_all(exp.Identifier):
            node.set("this", node.this.strip())

        # 4. Export to a standardized string format
        # identify=False removes "quotes" from table/column names
        canonical = tree.sql(identify=False, comments=False)

        # 5. Final cleanup: remove syntax marks to ensure 100% string parity
        return canonical.replace('"', '').replace("'", "").replace(";", "").strip()

    except Exception:
        # If the model hallucinates complete gibberish that won't parse,
        # fallback to a basic regex strip so the code doesn't crash.
        return re.sub(r'[\"\';]', '', sql_string.lower()).strip()

def evaluate(sample):
    # Setup Prompt
    prompt = pipe.apply_chat_template(sample["messages"][:2], tokenize=False, add_generation_prompt=True)
    outputs = pipe(prompt, max_new_tokens=256)

    # Isolate prediction
    full_output = outputs[0]['generated_text']
    predicted_answer = full_output[len(prompt):].strip()

    # Extract and Compare
    predicted_sql = extract_sql(predicted_answer)
    original_sql = sample["messages"][2]["content"]

    # Logical Comparison
    gold_canonical = logical_norm(original_sql)
    pred_canonical = logical_norm(predicted_sql)

    is_match = (gold_canonical == pred_canonical) and (pred_canonical != "")

    print(f"Question: {sample['messages'][1]['content']}")
    print(f"Original SQL: {original_sql}")
    print(f"Predicted SQL: {predicted_sql}")
    print(f"Match: {is_match}")
    print("-" * 80)

    return 1 if is_match else 0

# --- Execution ---
pipe = TextToSQLPipeline(model, tokenizer)

print("Loading test dataset...")
eval_dataset = load_dataset("json", data_files="test_dataset.json", split="train")
number_of_eval_samples = min(1000, len(eval_dataset))

success_rate = []
print(f"Evaluating on {number_of_eval_samples} samples...")

for s in tqdm(eval_dataset.shuffle(seed=42).select(range(number_of_eval_samples))):
    success_rate.append(evaluate(s))

# Final Stats
accuracy = sum(success_rate) / len(success_rate)
print(f"\n{'='*60}")
print(f"ACCURACY: {accuracy*100:.2f}%")
print(f"Correct predictions: {sum(success_rate)}/{number_of_eval_samples}")
print(f"{'='*60}")

Loading test dataset...
Evaluating on 1000 samples...


  0%|          | 1/1000 [00:21<6:03:20, 21.82s/it]

Question: How did the jab tak hai jaan movie gross worldwide?
Original SQL: SELECT worldwide_gross FROM table_name_60 WHERE movie = "jab tak hai jaan"
Predicted SQL: SELECT worldwide_gross FROM table_name_60 WHERE movie = "jab tak hai jaan"
Match: True
--------------------------------------------------------------------------------


  0%|          | 2/1000 [00:43<5:57:43, 21.51s/it]

Question: Who is the player with a t7 place?
Original SQL: SELECT player FROM table_name_56 WHERE place = "t7"
Predicted SQL: SELECT player FROM table_name_56 WHERE place = "t7"
Match: True
--------------------------------------------------------------------------------


...

Question: Which episodein season 3 had 175020 for a production code?
Original SQL: SELECT _number FROM table_2602958_4 WHERE prod_code = 175020
Predicted SQL: SELECT _number FROM table_2602958_4 WHERE prod_code = 175020
Match: True
--------------------------------------------------------------------------------


100%|█████████▉| 999/1000 [5:47:20<00:21, 21.04s/it]

Question: What is the result for November 25, 2001?
Original SQL: SELECT result FROM table_name_45 WHERE date = "november 25, 2001"
Predicted SQL: SELECT result FROM table_name_45 WHERE date = "november 25, 2001"
Match: True
--------------------------------------------------------------------------------


100%|██████████| 1000/1000 [5:47:41<00:00, 20.86s/it]

Question: Tell me the games for total less than 4 for swimming 1968
Original SQL: SELECT games FROM table_name_87 WHERE total < 4 AND sport = "swimming" AND years = "1968"
Predicted SQL: SELECT games FROM table_name_87 WHERE total < 4 AND sport = "swimming" AND years = "1968"
Match: True
--------------------------------------------------------------------------------

ACCURACY: 86.50%
Correct predictions: 865/1000
